Author: Bryan Sandor

Course: Stat 554

Title: Final Project

# Preamble

For this project, we'll use a `Jupyter` notebook fitting a machine learning model using `pyspark`'s `MLib` module. We'll use code to read in a stream of data and use that model to perform predictions on the stream, outputing them to the console. The data is modified from the UCI machine learning repository and studies the relationship between power consumption in Tetouan City and various factors like time of day, temperature, and humidity.

We begin by importing all the necessary libraries.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA

# Part 1: Fitting the Model

## Reading in the Data

We will read the data in using a `pandas` data frame, initially.

In [2]:
powerData = pd.read_csv("power_ml_data.csv")

Then we convert it to a `spark` data frame, first initializing our `spark` session.

In [3]:
spark = SparkSession.builder \
        .appName("proj2") \
        .config("spark.sql.ansi.enabled", "false") \
        .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 14:43:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
powerDF = spark.createDataFrame(powerData)
powerDF.show(n = 10, truncate = False)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|6.559      |73.8    |0.083     |0.051                |0.119        |34055.6962  |16128.87538 |20240.96386 |1    |0   |
|6.414      |74.5    |0.083     |0.07                 |0.085        |29814.68354 |19375.07599 |20131.08434 |1    |0   |
|6.313      |74.5    |0.08      |0.062                |0.1          |29128.10127 |19006.68693 |19668.43373 |1    |0   |
|6.121      |75.0    |0.083     |0.091                |0.096        |28228.86076 |18361.09422 |18899.27711 |1    |0   |
|5.921      |75.7    |0.081     |0.048                |0.085        |27335.6962  |17872.34043 |18442.40964 |1    |0   |
|5.853      |76.9    |0.081     |0.059  

For this assignment, we will use `Power_Zone_3` as the response and the other variables as predictors.

## Pipeline: Manipulations and Formatting

Now we need to make some adjustments to the data.

### Change `Hour` Class

In [5]:
powerDF.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

First, we need to re-cast the `Hour` column as `DoubleType` instead of `LongType()`.

In [6]:
recast = SQLTransformer(statement="SELECT *, CAST(Hour AS DOUBLE) AS db_hour FROM __THIS__")

recast.transform(powerDF).schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True), StructField('db_hour', DoubleType(), True)])

### Binarize `Hour` Column

Now that the `Hour` column is appropriately cast, we binarize it with the margin at $6.5$, i.e., night vs day.

In [7]:
binaryTrans = Binarizer(threshold = 6.5, inputCol = "db_hour", outputCol = "bin_hour")

binaryTrans.transform(
    recast.transform(powerDF)
).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|    0.0|     0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|    0.0|     0.0|
|      5.921|    75.7|     0.081|        

### One-Hot Encode the `Month` Column

Next, we will one-hot encode the `Month` column

In [8]:
encodeTrans = OneHotEncoder(inputCol = "Month", outputCol = "enc_month")

encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|     enc_month|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|    0.0|     0.0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361

### PCA fit

To perform a PCA fit, we need to map several columns into a single column, which we will accomplish using the `VectorAssembler()`, combining `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows`.

In [9]:
assemblerPCA = VectorAssembler(inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], outputCol = "PC")

In [10]:
assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|     enc_month|                  PC|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|    0.0|     0.0

Now we perform the PCA fit.

In [11]:
pca = PCA(k = 2, inputCol = "PC")
pca.setOutputCol("pca_features")
pcaTrans = pca.fit(
    assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
    )
)

In [12]:
pcaTrans.transform(
    assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
    )
).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+--------------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|     enc_month|                  PC|        pca_features|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+--------------+--------------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|[1.79440486365695...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|[1.80604083009823...|
|      6.313|    74.5|      0.

### Response and Predictors

To prepare the data for analysis, we begin by setting `Power_Zone_3` as our response by renaming it `label`.

In [13]:
labelTrans = SQLTransformer(
    statement = """
                SELECT pca_features, bin_hour, Power_Zone_1, Power_Zone_2, enc_month, Power_Zone_3 as label FROM __THIS__
                """
)

labelTrans.transform(
    pcaTrans.transform(
    assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
    )
    )
).show()

+--------------------+--------+------------+------------+--------------+-----------+
|        pca_features|bin_hour|Power_Zone_1|Power_Zone_2|     enc_month|      label|
+--------------------+--------+------------+------------+--------------+-----------+
|[1.79440486365695...|     0.0|  34055.6962| 16128.87538|(12,[1],[1.0])|20240.96386|
|[1.80604083009823...|     0.0| 29814.68354| 19375.07599|(12,[1],[1.0])|20131.08434|
|[1.81022976305638...|     0.0| 29128.10127| 19006.68693|(12,[1],[1.0])|19668.43373|
|[1.79866765174088...|     0.0| 28228.86076| 18361.09422|(12,[1],[1.0])|18899.27711|
|[1.86328720163797...|     0.0|  27335.6962| 17872.34043|(12,[1],[1.0])|18442.40964|
|[1.87820674500461...|     0.0| 26624.81013| 17416.41337|(12,[1],[1.0])|18130.12048|
|[1.91529298717955...|     0.0| 25998.98734| 16993.31307|(12,[1],[1.0])|17945.06024|
|[1.92400540807029...|     0.0| 25446.07595| 16661.39818|(12,[1],[1.0])|17459.27711|
|[1.89501930353027...|     0.0| 24777.72152| 16227.35562|(12,[1],

Finally, we combine the predictors into a single column named `features`.

In [14]:
assemblerFeatures = VectorAssembler(inputCols = ["pca_features", "bin_hour", "Power_Zone_1", "Power_Zone_2", "enc_month"], outputCol = "features")

assemblerFeatures.transform(
    labelTrans.transform(
    pcaTrans.transform(
    assemblerPCA.transform(
    encodeTrans.fit(powerDF).transform(
    binaryTrans.transform(
        recast.transform(powerDF)
    )
    )
    )
    )
    )
).show()

+--------------------+--------+------------+------------+--------------+-----------+--------------------+
|        pca_features|bin_hour|Power_Zone_1|Power_Zone_2|     enc_month|      label|            features|
+--------------------+--------+------------+------------+--------------+-----------+--------------------+
|[1.79440486365695...|     0.0|  34055.6962| 16128.87538|(12,[1],[1.0])|20240.96386|(17,[0,1,3,4,6],[...|
|[1.80604083009823...|     0.0| 29814.68354| 19375.07599|(12,[1],[1.0])|20131.08434|(17,[0,1,3,4,6],[...|
|[1.81022976305638...|     0.0| 29128.10127| 19006.68693|(12,[1],[1.0])|19668.43373|(17,[0,1,3,4,6],[...|
|[1.79866765174088...|     0.0| 28228.86076| 18361.09422|(12,[1],[1.0])|18899.27711|(17,[0,1,3,4,6],[...|
|[1.86328720163797...|     0.0|  27335.6962| 17872.34043|(12,[1],[1.0])|18442.40964|(17,[0,1,3,4,6],[...|
|[1.87820674500461...|     0.0| 26624.81013| 17416.41337|(12,[1],[1.0])|18130.12048|(17,[0,1,3,4,6],[...|
|[1.91529298717955...|     0.0| 25998.98734| 1

## Elastic Net Model